In [ ]:
# ============================================
# PARTIE 1 : IMPORTATION DES DONNÉES ET EXPLORATION INITIALE
# ============================================

# 1.1 Importation des bibliothèques nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Pour le modèle LSTM (sera utilisé dans les parties suivantes)
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Configuration des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("Bibliothèques importées avec succès!")
print(f"Version de pandas: {pd.__version__}")
print(f"Version de numpy: {np.__version__}")

# 1.2 Chargement de l'ensemble de données
print("\n" + "="*50)
print("CHARGEMENT DU DATASET")
print("="*50)

# Note: Téléchargez d'abord le fichier depuis: 
# https://archive.ics.uci.edu/ml/datasets/individual+household+electric+power+consumption
data_path = '/content/household_power_consumption.txt'

try:
    df = pd.read_csv(data_path, sep=';', 
                     parse_dates={'datetime': ['Date', 'Time']},
                     infer_datetime_format=True,
                     low_memory=False,
                     na_values=['?'])
    print("Fichier chargé depuis le chemin local.")
except FileNotFoundError:
    # Alternative: télécharger directement depuis UCI
    import urllib.request
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
    import zipfile
    import io
    
    print("Téléchargement du dataset depuis UCI...")
    with urllib.request.urlopen(url) as response:
        with zipfile.ZipFile(io.BytesIO(response.read())) as zip_file:
            with zip_file.open('household_power_consumption.txt') as file:
                df = pd.read_csv(file, sep=';', 
                               parse_dates={'datetime': ['Date', 'Time']},
                               infer_datetime_format=True,
                               low_memory=False,
                               na_values=['?'])
    print("Téléchargement terminé!")

# 1.3 Affichage des premières lignes
print("\n" + "="*50)
print("APERÇU DES DONNÉES")
print("="*50)
print("Premières lignes du dataset:")
print(df.head())

print("\nDernières lignes du dataset:")
print(df.tail())

# 1.4 Structure du dataset
print("\n" + "="*50)
print("STRUCTURE DU DATASET")
print("="*50)
print("Informations générales:")
print(df.info())

print(f"\nNombre total d'enregistrements: {len(df):,}")
print(f"Nombre de colonnes: {len(df.columns)}")
print(f"Période: de {df['datetime'].min()} à {df['datetime'].max()}")

# 1.5 Types de données
print("\n" + "="*50)
print("TYPES DE DONNÉES")
print("="*50)
print("Types de données par colonne:")
print(df.dtypes)

# Vérification des types après conversion
print("\nVérification des types:")
for col in df.columns:
    if col != 'datetime':
        print(f"{col}: {df[col].dtype}")

# 1.6 Statistiques descriptives
print("\n" + "="*50)
print("STATISTIQUES DESCRIPTIVES")
print("="*50)
print("Statistiques pour les colonnes numériques:")
print(df.describe())

# Statistiques supplémentaires
print("\nStatistiques supplémentaires:")
print(f"Valeurs uniques dans 'Global_active_power': {df['Global_active_power'].nunique()}")
print(f"Valeurs nulles dans 'Global_active_power': {df['Global_active_power'].isnull().sum()}")
print(f"Valeurs nulles totales: {df.isnull().sum().sum()}")

# 1.7 Analyse des colonnes
print("\n" + "="*50)
print("ANALYSE DES COLONNES")
print("="*50)
print("Description des colonnes:")
print("""
- datetime: Date et heure de la mesure (format: JJ/MM/AAAA HH:MM:SS)
- Global_active_power: Puissance active globale (en watts)
- Global_reactive_power: Puissance réactive globale (en vars)
- Voltage: Tension (en volts)
- Global_intensity: Intensité globale (en ampères)
- Sub_metering_1: Sous-compteur 1 (énergie en watt-heures)
- Sub_metering_2: Sous-compteur 2 (énergie en watt-heures)
- Sub_metering_3: Sous-compteur 3 (énergie en watt-heures)
""")

# 1.8 Visualisation de la distribution des données
print("\n" + "="*50)
print("VISUALISATION EXPLORATOIRE")
print("="*50)

# Distribution des variables principales
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Distribution de Global_active_power
axes[0, 0].hist(df['Global_active_power'].dropna(), bins=50, color='blue', alpha=0.7, edgecolor='black')
axes[0, 0].set_title('Distribution de Global_active_power')
axes[0, 0].set_xlabel('Puissance (watts)')
axes[0, 0].set_ylabel('Fréquence')
axes[0, 0].grid(True, alpha=0.3)

# Distribution de Voltage
axes[0, 1].hist(df['Voltage'].dropna(), bins=50, color='green', alpha=0.7, edgecolor='black')
axes[0, 1].set_title('Distribution de Voltage')
axes[0, 1].set_xlabel('Tension (volts)')
axes[0, 1].set_ylabel('Fréquence')
axes[0, 1].grid(True, alpha=0.3)

# Distribution de Global_intensity
axes[1, 0].hist(df['Global_intensity'].dropna(), bins=50, color='red', alpha=0.7, edgecolor='black')
axes[1, 0].set_title('Distribution de Global_intensity')
axes[1, 0].set_xlabel('Intensité (ampères)')
axes[1, 0].set_ylabel('Fréquence')
axes[1, 0].grid(True, alpha=0.3)

# Distribution de Sub_metering_3
axes[1, 1].hist(df['Sub_metering_3'].dropna(), bins=50, color='purple', alpha=0.7, edgecolor='black')
axes[1, 1].set_title('Distribution de Sub_metering_3')
axes[1, 1].set_xlabel('Énergie (watt-heures)')
axes[1, 1].set_ylabel('Fréquence')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 1.9 Analyse temporelle
print("\n" + "="*50)
print("ANALYSE TEMPORELLE")
print("="*50)

# Créer une copie avec l'index datetime pour l'analyse temporelle
df_temp = df.copy()
df_temp.set_index('datetime', inplace=True)

# Échantillonnage mensuel pour visualisation
monthly_avg = df_temp['Global_active_power'].resample('M').mean()

# Visualisation de la tendance mensuelle
plt.figure(figsize=(15, 6))
plt.plot(monthly_avg.index, monthly_avg.values, color='blue', linewidth=2)
plt.title('Consommation électrique moyenne mensuelle')
plt.xlabel('Date')
plt.ylabel('Consommation moyenne (watts)')
plt.grid(True, alpha=0.3)
plt.show()

# Détection des tendances saisonnières
print("\nTendance saisonnière:")
print("-" * 30)
print("Consommation par mois (moyenne):")
monthly_stats = df_temp.groupby(df_temp.index.month)['Global_active_power'].agg(['mean', 'std'])
print(monthly_stats)

# 1.10 Corrélation entre les variables
print("\n" + "="*50)
print("ANALYSE DES CORRÉLATIONS")
print("="*50)

# Calcul de la matrice de corrélation
correlation_matrix = df[['Global_active_power', 'Global_reactive_power', 
                         'Voltage', 'Global_intensity', 
                         'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].corr()

# Visualisation de la matrice de corrélation
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            fmt='.2f', square=True, linewidths=0.5)
plt.title('Matrice de corrélation des variables')
plt.tight_layout()
plt.show()

# Identification des corrélations fortes
print("\nCorrélations fortes (> 0.8):")
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr = correlation_matrix.iloc[i, j]
        if abs(corr) > 0.8:
            print(f"- {correlation_matrix.columns[i]} et {correlation_matrix.columns[j]}: {corr:.3f}")

# 1.11 Détection des valeurs aberrantes
print("\n" + "="*50)
print("DÉTECTION DES VALEURS ABERRANTES")
print("="*50)

# Box plots pour identifier les outliers
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Global_active_power
sns.boxplot(y=df['Global_active_power'].dropna(), ax=axes[0], color='skyblue')
axes[0].set_title('Boxplot - Global_active_power')
axes[0].set_ylabel('Puissance (watts)')

# Voltage
sns.boxplot(y=df['Voltage'].dropna(), ax=axes[1], color='lightgreen')
axes[1].set_title('Boxplot - Voltage')
axes[1].set_ylabel('Tension (volts)')

# Global_intensity
sns.boxplot(y=df['Global_intensity'].dropna(), ax=axes[2], color='lightcoral')
axes[2].set_title('Boxplot - Global_intensity')
axes[2].set_ylabel('Intensité (ampères)')

plt.tight_layout()
plt.show()

# Statistiques sur les outliers
print("\nNombre d'outliers (méthode IQR):")
for col in ['Global_active_power', 'Voltage', 'Global_intensity']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"- {col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.2f}%)")

# 1.12 Résumé de l'exploration
print("\n" + "="*50)
print("RÉSUMÉ DE L'EXPLORATION")
print("="*50)
print(f"""
1. Dataset chargé avec succès:
   - {len(df):,} enregistrements
   - {len(df.columns)} colonnes
   - Période: {df['datetime'].min()} à {df['datetime'].max()}

2. Variables principales:
   - Consommation électrique: {df['Global_active_power'].min():.2f} à {df['Global_active_power'].max():.2f} watts
   - Tension: {df['Voltage'].min():.2f} à {df['Voltage'].max():.2f} volts
   - Intensité: {df['Global_intensity'].min():.2f} à {df['Global_intensity'].max():.2f} ampères

3. Qualité des données:
   - Valeurs manquantes: {df.isnull().sum().sum():,} ({df.isnull().sum().sum()/len(df)*100:.2f}%)
   - Colonnes avec valeurs manquantes: {sum(df.isnull().any())}

4. Corrélations fortes:
   - Global_active_power et Global_intensity: {correlation_matrix.loc['Global_active_power', 'Global_intensity']:.3f}
   - Sub_metering_1 et Sub_metering_2: {correlation_matrix.loc['Sub_metering_1', 'Sub_metering_2']:.3f}

5. Prochaines étapes:
   - Partie 2: Gestion des valeurs manquantes
   - Partie 3: Visualisation avancée
   - Partie 4: Prétraitement pour LSTM
   - Partie 5-6: Modélisation et évaluation
""")
# ============================================
# PARTIE 2 : GESTION DES VALEURS MANQUANTES
# ============================================
# (Suite du code fourni dans votre notebook)

# Identification des valeurs manquantes
print("\n" + "="*50)
print("PARTIE 2 : GESTION DES VALEURS MANQUANTES")
print("="*50)

print("Valeurs manquantes par colonne:")
missing_values = df.isnull().sum()
print(missing_values)

print(f"\nPourcentage de valeurs manquantes:")
print((missing_values / len(df)) * 100)

# Visualisation des valeurs manquantes
plt.figure(figsize=(12, 6))
plt.bar(missing_values.index, missing_values.values)
plt.title('Nombre de valeurs manquantes par colonne')
plt.xlabel('Colonnes')
plt.ylabel('Nombre de valeurs manquantes')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Complétion des valeurs manquantes par la moyenne
print("\nComplétion des valeurs manquantes par la moyenne...")
for col in df.columns:
    if col != 'datetime':  # Ne pas traiter la colonne datetime
        if df[col].isnull().sum() > 0:
            mean_value = df[col].mean()
            df[col].fillna(mean_value, inplace=True)
            print(f"Colonne '{col}': {df[col].isnull().sum()} valeurs manquantes restantes")

# Vérification finale
print("\nVérification finale des valeurs manquantes:")
print(df.isnull().sum().sum(), "valeurs manquantes dans tout le dataset")

# S'assurer que les colonnes numériques sont bien au bon type
for col in df.columns:
    if col != 'datetime':
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Sauvegarder les données nettoyées
df.to_csv('household_power_consumption_cleaned.csv', index=False)
print("\nDonnées nettoyées sauvegardées!")

# ============================================
# PARTIE 3 : VISUALISATION DES DONNÉES
# ============================================
print("\n" + "="*50)
print("PARTIE 3 : VISUALISATION DES DONNÉES")
print("="*50)

# Assurer que la colonne datetime est bien l'index
df.set_index('datetime', inplace=True)

# Rééchantillonnage de 'Global_active_power' par jour
daily_power = df['Global_active_power'].resample('D').agg(['sum', 'mean'])

# Visualisation de la somme et de la moyenne journalière
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10))

# Somme journalière
ax1.plot(daily_power.index, daily_power['sum'], color='blue', linewidth=1)
ax1.set_title('Consommation électrique totale par jour (Global_active_power)')
ax1.set_xlabel('Date')
ax1.set_ylabel('Somme (watt-heures)')
ax1.grid(True, alpha=0.3)

# Moyenne journalière
ax2.plot(daily_power.index, daily_power['mean'], color='green', linewidth=1)
ax2.set_title('Consommation électrique moyenne par jour (Global_active_power)')
ax2.set_xlabel('Date')
ax2.set_ylabel('Moyenne (watts)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Rééchantillonnage de 'Global_intensity' par jour
daily_intensity = df['Global_intensity'].resample('D').agg(['mean', 'std'])

# Visualisation de la moyenne et de l'écart type
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10))

# Moyenne journalière
ax1.plot(daily_intensity.index, daily_intensity['mean'], color='red', linewidth=1)
ax1.set_title('Intensité électrique moyenne par jour (Global_intensity)')
ax1.set_xlabel('Date')
ax1.set_ylabel('Moyenne (ampères)')
ax1.grid(True, alpha=0.3)

# Écart type journalier
ax2.plot(daily_intensity.index, daily_intensity['std'], color='purple', linewidth=1)
ax2.set_title('Écart type de l\'intensité électrique par jour (Global_intensity)')
ax2.set_xlabel('Date')
ax2.set_ylabel('Écart type (ampères)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Analyse supplémentaire : visualisation de la consommation par mois
df_copy = df.copy()
df_copy['Month'] = df_copy.index.month
df_copy['Year'] = df_copy.index.year

plt.figure(figsize=(12, 6))
df_copy.boxplot(column='Global_active_power', by='Month', grid=True)
plt.title('Distribution de la consommation par mois')
plt.suptitle('')
plt.xlabel('Mois')
plt.ylabel('Consommation (watts)')
plt.tight_layout()
plt.show()

# ============================================
# PARTIE 4 : PRÉTRAITEMENT DES DONNÉES POUR LSTM
# ============================================
print("\n" + "="*50)
print("PARTIE 4 : PRÉTRAITEMENT DES DONNÉES POUR LSTM")
print("="*50)

# Sélection des colonnes pertinentes
# Nous allons utiliser toutes les colonnes pour la prédiction
features = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
            'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

# Création du dataset pour LSTM
data = df[features].copy()

# Normalisation des données
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)

print(f"Dimensions des données normalisées: {data_scaled.shape}")
print(f"Valeurs min après normalisation: {data_scaled.min(axis=0)}")
print(f"Valeurs max après normalisation: {data_scaled.max(axis=0)}")

# Fonction pour créer des séquences
def create_sequences(data, sequence_length):
    X, y = [], []
    for i in range(len(data) - sequence_length):
        X.append(data[i:i + sequence_length])
        y.append(data[i + sequence_length, 0])  # Prédire Global_active_power
    return np.array(X), np.array(y)

# Création des séquences
SEQUENCE_LENGTH = 60  # Utiliser 60 minutes de données passées pour prédire la prochaine

X, y = create_sequences(data_scaled, SEQUENCE_LENGTH)

print(f"Shape de X: {X.shape}")  # (échantillons, séquence, features)
print(f"Shape de y: {y.shape}")  # (échantillons,)

# Division en ensembles d'entraînement et de test
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print(f"Taille de l'ensemble d'entraînement: {X_train.shape[0]}")
print(f"Taille de l'ensemble de test: {X_test.shape[0]}")

# Vérification rapide
print(f"\nStatistiques de y_train:")
print(f"Min: {y_train.min()}, Max: {y_train.max()}, Moyenne: {y_train.mean():.4f}")
print(f"Statistiques de y_test:")
print(f"Min: {y_test.min()}, Max: {y_test.max()}, Moyenne: {y_test.mean():.4f}")

# ============================================
# PARTIE 5 : CONSTRUCTION DU MODÈLE LSTM
# ============================================
print("\n" + "="*50)
print("PARTIE 5 : CONSTRUCTION DU MODÈLE LSTM")
print("="*50)

# Importation des bibliothèques supplémentaires si nécessaire
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Construction du modèle LSTM
def create_lstm_model(input_shape):
    model = Sequential([
        # Première couche LSTM
        LSTM(100, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        
        # Deuxième couche LSTM
        LSTM(100, return_sequences=True),
        Dropout(0.2),
        
        # Troisième couche LSTM
        LSTM(50, return_sequences=True),
        Dropout(0.2),
        
        # Quatrième couche LSTM
        LSTM(50),
        Dropout(0.2),
        
        # Couche dense de sortie
        Dense(1)
    ])
    
    return model

# Création du modèle
model = create_lstm_model((X_train.shape[1], X_train.shape[2]))

# Compilation du modèle
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

# Affichage du résumé du modèle
print("Architecture du modèle LSTM:")
model.summary()

# Callbacks pour l'entraînement
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=0.00001,
        verbose=1
    )
]

print("\nParamètres d'entraînement:")
print(f"Nombre total de paramètres: {model.count_params():,}")
print(f"Callbacks: EarlyStopping, ReduceLROnPlateau")
print(f"Optimiseur: Adam")
print(f"Fonction de perte: MSE (Mean Squared Error)")

# ============================================
# PARTIE 6 : ENTRAÎNEMENT ET ÉVALUATION DU MODÈLE LSTM
# ============================================
print("\n" + "="*50)
print("PARTIE 6 : ENTRAÎNEMENT ET ÉVALUATION DU MODÈLE")
print("="*50)

# Entraînement du modèle
print("Début de l'entraînement...")
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=128,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
    verbose=1
)

print("Entraînement terminé!")

# Visualisation des courbes de perte et d'erreur
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Courbe de perte
ax1.plot(history.history['loss'], label='Perte entraînement')
ax1.plot(history.history['val_loss'], label='Perte validation')
ax1.set_title('Évolution de la perte du modèle')
ax1.set_xlabel('Époque')
ax1.set_ylabel('Perte (MSE)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Courbe d'erreur MAE
ax2.plot(history.history['mae'], label='MAE entraînement')
ax2.plot(history.history['val_mae'], label='MAE validation')
ax2.set_title('Évolution de l\'erreur MAE')
ax2.set_xlabel('Époque')
ax2.set_ylabel('MAE')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Évaluation du modèle sur les données de test
train_loss, train_mae = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)

print(f"\nPerformance du modèle:")
print(f"Erreur sur l'ensemble d'entraînement:")
print(f"  - MSE: {train_loss:.6f}")
print(f"  - MAE: {train_mae:.6f}")
print(f"\nErreur sur l'ensemble de test:")
print(f"  - MSE: {test_loss:.6f}")
print(f"  - MAE: {test_mae:.6f}")

# Prédictions sur les données de test
y_pred = model.predict(X_test)

# Dénormalisation des prédictions et des valeurs réelles
# Créer un tableau avec les mêmes dimensions que les données originales
def inverse_transform_predictions(y, features_scaled):
    # Créer une matrice de zéros avec la même forme que les données normalisées
    dummy = np.zeros((len(y), data_scaled.shape[1]))
    dummy[:, 0] = y  # Mettre les prédictions dans la première colonne
    return scaler.inverse_transform(dummy)[:, 0]

y_test_actual = inverse_transform_predictions(y_test, data_scaled)
y_pred_actual = inverse_transform_predictions(y_pred.flatten(), data_scaled)

# Visualisation des prédictions vs réalité
plt.figure(figsize=(15, 6))
plt.plot(y_test_actual[:500], label='Valeurs réelles', alpha=0.7)
plt.plot(y_pred_actual[:500], label='Prédictions', alpha=0.7)
plt.title('Comparaison des prédictions vs valeurs réelles (500 premiers échantillons)')
plt.xlabel('Échantillons')
plt.ylabel('Consommation électrique (watts)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Graphique des résidus
residuals = y_test_actual - y_pred_actual
plt.figure(figsize=(15, 6))
plt.scatter(y_pred_actual, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.title('Graphique des résidus')
plt.xlabel('Valeurs prédites')
plt.ylabel('Résidus')
plt.grid(True, alpha=0.3)
plt.show()

# Distribution des erreurs
plt.figure(figsize=(12, 5))
plt.hist(residuals, bins=50, edgecolor='black', alpha=0.7)
plt.title('Distribution des erreurs de prédiction')
plt.xlabel('Erreur (watts)')
plt.ylabel('Fréquence')
plt.grid(True, alpha=0.3)
plt.show()

# Métriques supplémentaires
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test_actual, y_pred_actual)
mse = mean_squared_error(y_test_actual, y_pred_actual)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_actual, y_pred_actual)

print("\nMétriques d'évaluation détaillées:")
print(f"MAE (Erreur Absolue Moyenne): {mae:.4f} watts")
print(f"MSE (Erreur Quadratique Moyenne): {mse:.4f} watts²")
print(f"RMSE (Racine de l'Erreur Quadratique Moyenne): {rmse:.4f} watts")
print(f"R² (Coefficient de Détermination): {r2:.4f}")

# Sauvegarde du modèle
model.save('lstm_power_consumption_model.h5')
print("\nModèle sauvegardé!")

# Fonction pour prédire la consommation future
def predict_future(model, last_sequence, steps=24):
    predictions = []
    current_sequence = last_sequence.copy()
    
    for _ in range(steps):
        # Faire la prédiction
        pred = model.predict(current_sequence.reshape(1, current_sequence.shape[0], current_sequence.shape[1]), verbose=0)
        predictions.append(pred[0, 0])
        
        # Mettre à jour la séquence
        new_row = current_sequence[-1].copy()
        new_row[0] = pred[0, 0]  # Mettre la nouvelle prédiction
        current_sequence = np.roll(current_sequence, -1, axis=0)
        current_sequence[-1] = new_row
    
    return np.array(predictions)

# Prédiction des 24 prochaines heures
last_sequence = X_test[-1]
future_predictions = predict_future(model, last_sequence, steps=24)
future_predictions_actual = inverse_transform_predictions(future_predictions, data_scaled)

print("\nPrédictions des 24 prochaines heures:")
for i, pred in enumerate(future_predictions_actual):
    print(f"Heure {i+1}: {pred:.4f} watts")

# ============================================
# RÉSUMÉ FINAL
# ============================================
print("\n" + "="*50)
print("RÉSUMÉ DU PROJET")
print("="*50)
print(f"""
1. Dataset: Household Power Consumption
   - {len(df)} enregistrements
   - Période: {df.index.min()} à {df.index.max()}
   - {len(features)} variables prédictives

2. Prétraitement:
   - Valeurs manquantes: remplacées par la moyenne
   - Normalisation: MinMaxScaler [-1, 1]
   - Séquences créées: {SEQUENCE_LENGTH} pas de temps

3. Architecture LSTM:
   - 4 couches LSTM avec Dropout
   - {model.count_params():,} paramètres
   - Optimiseur: Adam
   - Fonction de perte: MSE

4. Performance:
   - MAE: {mae:.4f} watts
   - RMSE: {rmse:.4f} watts
   - R²: {r2:.4f}

5. Capacités:
   - Prédiction de la consommation électrique
   - Analyse des tendances saisonnières
   - Détection des anomalies de consommation
""")

: 